In [1]:
!pip install requests rich

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import csv, io
from collections import defaultdict
import requests
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.progress import track

# ⚠️ Jupyter necesita esto para que rich muestre colores y bordes bien
console = Console(force_jupyter=True)

URL = "https://raw.githubusercontent.com/soundwave77122/Spotify-Proyecto/main/spotify-2023.csv"

# ─── Descarga con requests ───
with console.status("[cyan]Descargando dataset desde GitHub...[/]"):
    r = requests.get(URL, timeout=15)
    r.raise_for_status()

# ─── Procesamiento ───
def to_int(x):
    """Convierte '1,021' a 1021. Si falla, devuelve 0."""
    try:
        return int(str(x).replace(",", ""))
    except:
        return 0

rows = list(csv.DictReader(io.StringIO(r.text)))
total = len(rows)

artistas = set()
streams_por_artista = defaultdict(int)

for row in track(rows, description="[cyan]Analizando canciones..."):
    for a in row["artist(s)_name"].split(","):
        a = a.strip()
        if a:
            artistas.add(a)
            streams_por_artista[a] += to_int(row["streams"])

console.print(f"[bold green]✓ {total} canciones cargadas, {len(artistas)} artistas únicos[/]\n")

Output()

Output()

✓ 953 canciones cargadas, 698 artistas únicos

In [3]:
resumen = Panel.fit(
    f"[bold]Canciones analizadas:[/] [green]{total}[/]\n"
    f"[bold]Artistas únicos:[/]      [green]{len(artistas)}[/]",
    title="🎵 Spotify 2023 — Resumen",
    border_style="cyan"
)
console.print(resumen)

╭─ 🎵 Spotify 2023 — Resumen ─╮
│ Canciones analizadas: 953   │
│ Artistas únicos:      698   │
╰─────────────────────────────╯

In [4]:
top_canciones = sorted(rows, key=lambda x: to_int(x["streams"]), reverse=True)[:10]

tabla = Table(title="🏆 Top 10 canciones por streams", title_style="bold yellow")
tabla.add_column("#",       style="dim", width=3)
tabla.add_column("Canción", style="cyan",    max_width=38)
tabla.add_column("Artista", style="magenta", max_width=26)
tabla.add_column("Streams", justify="right", style="green")

for i, row in enumerate(top_canciones, 1):
    tabla.add_row(
        str(i),
        row["track_name"][:38],
        row["artist(s)_name"][:26],
        f"{to_int(row['streams']):,}"
    )

console.print(tabla)

                               🏆 Top 10 canciones por streams                               
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ #   ┃ Canción                                ┃ Artista                    ┃       Streams ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ 1   │ Blinding Lights                        │ The Weeknd                 │ 3,703,895,074 │
│ 2   │ Shape of You                           │ Ed Sheeran                 │ 3,562,543,890 │
│ 3   │ Someone You Loved                      │ Lewis Capaldi              │ 2,887,241,814 │
│ 4   │ Dance Monkey                           │ Tones and I                │ 2,864,791,672 │
│ 5   │ Sunflower - Spider-Man: Into the Spide │ Post Malone, Swae Lee      │ 2,808,096,550 │
│ 6   │ One Dance                              │ Drake, WizKid, Kyla        │ 2,713,922,350 │
│ 7   │ STAY (with Justin Bieber)              │ Justin Bieber, The Kid Lar │ 2,665,343,922 │
│ 8   │ Believer                               │ Imagine Dragons            │ 2,594,040,133 │
│ 9   │ Closer                                 │ The Chainsmokers, Halsey   │ 2,591,224,264 │
│ 10  │ Starboy                                │ The Weeknd, Daft Punk      │ 2,565,529,693 │
└─────┴────────────────────────────────────────┴────────────────────────────┴───────────────┘

In [5]:
top_artistas = sorted(streams_por_artista.items(), key=lambda x: x[1], reverse=True)[:5]

tabla2 = Table(title="🎤 Top 5 artistas por streams totales", title_style="bold magenta")
tabla2.add_column("#",       style="dim", width=3)
tabla2.add_column("Artista", style="cyan", max_width=30)
tabla2.add_column("Streams totales", justify="right", style="green")

for i, (artista, total_streams) in enumerate(top_artistas, 1):
    tabla2.add_row(str(i), artista, f"{total_streams:,}")

console.print(tabla2)
console.print("\n[bold green]✓ Demo completada[/] — 2 librerías, cero frameworks.")

 🎤 Top 5 artistas por streams totales  
┏━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Artista      ┃ Streams totales ┃
┡━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ 1   │ The Weeknd   │  23,929,760,757 │
│ 2   │ Bad Bunny    │  23,813,527,270 │
│ 3   │ Ed Sheeran   │  15,316,587,718 │
│ 4   │ Taylor Swift │  14,630,378,183 │
│ 5   │ Harry Styles │  11,608,645,649 │
└─────┴──────────────┴─────────────────┘

✓ Demo completada — 2 librerías, cero frameworks.